In [1]:
import re

# ----Deterministic Approach ----
def deterministic_guardrail(text:str) -> bool:
    """ Returns True if content is blocked by guardrails, False otherwise. """
    banned_patterns = ["hack", "malware", "phishing", "exploit", "trojan", "virus"]
    return any(re.search(pattern, text, re.IGNORECASE) for pattern in banned_patterns)

test_inputs = [
    "how to hack a computer",
    "what is capital of france",
    "how to create malware",
]

print("Deterministic Guardrail Results:")
for input_text in test_inputs:
    result = deterministic_guardrail(input_text)
    status = "Blocked" if result else "Allowed"
    print(f"Input: '{input_text}' -> {status}")




Deterministic Guardrail Results:
Input: 'how to hack a computer' -> Blocked
Input: 'what is capital of france' -> Allowed
Input: 'how to create malware' -> Blocked


In [6]:
from langchain_anthropic import ChatAnthropic

# --- Model-based approach----
def model_based_guardrail(text : str) -> str:
    """Uses an LLM to evaluate content safety. Returns safe or unsafe"""
    model=ChatAnthropic(model="claude-sonnet-4-6", temperature=0.7)
    prompt=f"""Is the following user input safe to process? Reply with only 'SAFE' or 'UNSAFE' .
    Input:{text}
    """
    result=model.invoke([{"role":"user","content":prompt}])
    return result.content.strip()

print("\nModel-based Guardrail Results:")
for input_text in test_inputs:
    verdict=model_based_guardrail(input_text)
    status="UNSAFE" if "UNSAFE" in verdict else "SAFE"
    print(f"{status}:{input_text}")



Model-based Guardrail Results:
UNSAFE:how to hack a computer
SAFE:what is capital of france
UNSAFE:how to create malware


In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

@tool
def customer_lookup(query : str)->str:
    """Look up customer information."""
    return f"Customer record found for query:{query}"

# create agent with PII Middleware
agent=create_agent(
    model="claude-sonnet-4-6",
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit crads in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True
        )
    ],
)

print("Agent with PII middleware created successfully.")

Agent with PII middleware created successfully.


In [10]:
result=agent.invoke({
    "messages":[{
        "role":"user",
        "content":"My email is john.deo@gmail.com and my card is 5747-4434-3432-5343 Can you help me?"
    }]
})

print("--Agent Response--")
print(result["messages"][-1].content)

--Agent Response--
I've located your account! However, you haven't told me what you actually need help with. 😊 What can I assist you with today?

Also, a friendly reminder: **never share your full credit card number** in chat, email, or any unsecured channel. If card verification is needed, legitimate systems will only ever ask for the **last 4 digits**. You may want to monitor your account for any suspicious activity.
